## Deep Neural Network 
Build a deep neural network to classify MNIST digits

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [3]:
# Load MNIST and create train/test split

# Convert images to tensors and normalize with mean and std of MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
]) 

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Number of samples model processes at once during training/testing 
# Model sees 64 images at a time, computes loss and back propagates gradients, then SGD updates weights. 
batch_size = 64 

# Create data loaders to handle batching and shuffling of data during training/testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 60000
Test size: 10000


In [ ]:
model = nn.Sequential(
    nn.Flatten(),              # Convert 28x28 image tensors into 784-length 1D vectors, [Batch, 1, 28, 28] -> [Batch, 784]
    nn.Linear(784, 64),        # Hidden layer 1 with 64 neurons, batch normalization and ReLU activation 
    nn.BatchNorm1d(64),
    nn.ReLU(),

                        
    nn.Linear(64, 64),         # Hidden layer 2 with 64 neurons
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 64),         # Hidden layer 3 with 64 neurons
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.2),           # dropout rate to prevent overfitting

    nn.Linear(64, 10),         # Output layer for 10 MNIST classes
    nn.Softmax(dim=1)          # Convert logits to class probabilities
)

In [5]:
print(model)
total_params = sum(p.numel() for p in model.parameters())
print("Total model parameters:", total_params)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=10, bias=True)
  (6): Softmax(dim=1)
)
Total model parameters: 55050


In [6]:
criterion = nn.CrossEntropyLoss() # Cross-entropy loss compares predicted logits to true class indices, penalizing incorrect predictions more heavily
optimizer = optim.SGD(model.parameters(), lr=0.01) # Gradient Descent optimizer updates model weights based on computed gradients and learning rate

In [7]:
# Model training loop with GPU support if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

# epoch = one full pass through the entire training dataset (all batches)
# Steps/iterations per epoch = (dataset_size / batch_size)
# Each step: forward pass on one batch -> compute loss -> backward pass -> optimizer.step()

epochs = 30
for epoch in range(epochs):
    model.train() # Set model to training mode (enables dropout, batch norm, etc. if present)
    running_loss = 0.0 # Cumulative loss for the current epoch, used to compute average loss at the end of the epoch
    correct = 0 # Number of correct predictions for the current epoch, used to compute training accuracy
    total = 0 # Total number of samples processed in the current epoch, used to compute training accuracy

    # Iterate over mini-batches from the training loader
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        targets = nn.functional.one_hot(labels, num_classes=10).float() # Convert labels to one-hot encoding 

        optimizer.zero_grad() # Clear previous gradients before backpropagation
        outputs = model(images) # Forward pass: compute predicted probabilities for the current batch
        loss = criterion(outputs, targets) # Compute loss between predicted probabilities and one-hot encoded true labels
        loss.backward() # Backward pass: compute gradients of the loss with respect to model parameters
        optimizer.step() # Update model parameters based on computed gradients and learning rate

        # Track loss and accuracy for the current epoch
        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1) # Get predicted class 
        correct += (predicted == labels).sum().item() # Count correct predictions in the current batch
        total += labels.size(0)

    train_loss = running_loss / total # Epoch average loss per sample
    train_acc = correct / total # Epoch training accuracy 

    # Evaluate on the test split without computing gradients
    model.eval() # Set model to evaluation mode (disables dropout, batch norm updates, etc.)
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images) # Forward pass on test data to compute predicted probabilities
            predicted = outputs.argmax(dim=1) # Get predicted class
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    print(f"Epoch {epoch + 1}/{epochs} - loss: {train_loss:.4f} - train acc: {train_acc:.4f} - test acc: {test_acc:.4f}")

Using device: mps
Epoch 1/30 - loss: 2.2658 - train acc: 0.2468 - test acc: 0.3732
Epoch 2/30 - loss: 1.9568 - train acc: 0.5665 - test acc: 0.7167
Epoch 3/30 - loss: 1.7362 - train acc: 0.7794 - test acc: 0.8214
Epoch 4/30 - loss: 1.6642 - train acc: 0.8233 - test acc: 0.8347
Epoch 5/30 - loss: 1.6451 - train acc: 0.8328 - test acc: 0.8397
Epoch 6/30 - loss: 1.6359 - train acc: 0.8373 - test acc: 0.8435
Epoch 7/30 - loss: 1.6299 - train acc: 0.8408 - test acc: 0.8456
Epoch 8/30 - loss: 1.6256 - train acc: 0.8435 - test acc: 0.8495
Epoch 9/30 - loss: 1.6223 - train acc: 0.8454 - test acc: 0.8511
Epoch 10/30 - loss: 1.6194 - train acc: 0.8481 - test acc: 0.8528
Epoch 11/30 - loss: 1.6087 - train acc: 0.8604 - test acc: 0.8958
Epoch 12/30 - loss: 1.5721 - train acc: 0.9033 - test acc: 0.9113
Epoch 13/30 - loss: 1.5608 - train acc: 0.9126 - test acc: 0.9164
Epoch 14/30 - loss: 1.5544 - train acc: 0.9182 - test acc: 0.9189
Epoch 15/30 - loss: 1.5499 - train acc: 0.9213 - test acc: 0.9227
E